# Ensemble of Specialized Mixture of Experts (MoE) for NLI
This notebook implements a state-of-the-art Natural Language Inference pipeline combining:
1. **T5 Data Augmentation**: Generating synthetic hypotheses to improve robustness.
2. **POS-Specialized MoE**: A custom architecture with experts for Semantics, Entities, Actions, and Logic.
3. **Model Ensembling**: Averaging predictions from **DeBERTa-v3** and **ModernBERT** backbones.

In [7]:
!pip install datasets transformers torch spacy pandas numpy scikit-learn
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    TrainingArguments, 
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

nlp = spacy.load("en_core_web_sm", disable=["ner"])
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Data Augmentation (T5)
We use a T5 model to generate synthetic training examples to expand the diversity of our dataset.

In [13]:
def augment_dataset(df, n_samples=1000):
    gen_model_name = "t5-small"
    gen_tokenizer = T5Tokenizer.from_pretrained(gen_model_name)
    generator = T5ForConditionalGeneration.from_pretrained(gen_model_name).to(device)
    
    generator.eval()
    synthetic_examples = []
    subset = df.sample(n=min(n_samples, len(df)))
    
    print("Generating synthetic data...")
    for _, ex in subset.iterrows():
        prompt = f"generate hypothesis: premise: {ex['premise']} label: {ex['label']}"
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
        outputs = generator.generate(**inputs, max_length=64)
        gen_hyp = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
        synthetic_examples.append({"premise": ex["premise"], "hypothesis": gen_hyp, "label": ex["label"]})
    
    return pd.concat([df, pd.DataFrame(synthetic_examples)]).reset_index(drop=True)

# Load local data
try:
    train_df = pd.read_csv("training_data/NLI/train.csv")
    dev_df = pd.read_csv("training_data/NLI/dev.csv")
    augmented_train_df = augment_dataset(train_df)
except FileNotFoundError:
    print("CSV files not found. Please ensure training_data/train.csv exists.")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Generating synthetic data...


## 2. Multi-Backbone MoE Architecture
Each model in the ensemble utilizes a Mixture of Experts layer. This layer routes features to specialized heads based on POS-filtered text (Nouns for Entities, Verbs for Actions) and manual Logic features.

In [14]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h_size = self.encoder.config.hidden_size

        self.semantic_expert = nn.Linear(h_size, num_labels)
        self.entity_expert = nn.Linear(h_size, num_labels)
        self.action_expert = nn.Linear(h_size, num_labels)
        self.logic_expert = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, num_labels))

        self.gating = nn.Sequential(nn.Linear(h_size + 4, 64), nn.ReLU(), nn.Linear(64, 4))
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, entity_ids, action_ids, logic_features, labels=None):
        sem_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        ent_out = self.encoder(input_ids=entity_ids).last_hidden_state[:, 0, :]
        act_out = self.encoder(input_ids=action_ids).last_hidden_state[:, 0, :]

        l_sem, l_ent = self.semantic_expert(sem_out), self.entity_expert(ent_out)
        l_act, l_log = self.action_expert(act_out), self.logic_expert(logic_features.float())

        gate_input = torch.cat([sem_out, logic_features.float()], dim=1)
        gate_weights = torch.softmax(self.gating(gate_input), dim=1)

        final_logits = (gate_weights[:, 0:1] * l_sem + gate_weights[:, 1:2] * l_ent + 
                        gate_weights[:, 2:3] * l_act + gate_weights[:, 3:4] * l_log)

        loss = self.loss_fn(final_logits, labels) if labels is not None else None
        return {"loss": loss, "logits": final_logits}

## 3. Preprocessing and Feature Extraction
We extract linguistic features using Spacy to feed the MoE gating and specialized experts.

In [15]:
def get_pos_filtered_text(text, pos_tags):
    doc = nlp(str(text))
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

def make_preprocess_fn(tokenizer):
    def preprocess(example):
        main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)
        p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
        h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
        p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
        h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
        
        ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
        act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
        
        return {
            "input_ids": main_enc["input_ids"],
            "attention_mask": main_enc["attention_mask"],
            "entity_ids": ent_enc["input_ids"],
            "action_ids": act_enc["input_ids"],
            "logic_features": extract_logic_features(example["premise"], example["hypothesis"]),
            "label": int(example["label"])
        }
    return preprocess

## 4. Training and Ensembling
We train two separate MoE models (DeBERTa and ModernBERT) and then ensemble their outputs by averaging the logits.

In [16]:
backbones = {
    "deberta": "microsoft/deberta-v3-small",
    "modernbert": "answerdotai/ModernBERT-base"
}

all_logits = {}

for name, path in backbones.items():
    print(f"\n--- Training MoE with {name} backbone ---")
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = POSSpecializedMoE(path).to(device)
    
    prep_fn = make_preprocess_fn(tokenizer)
    train_ds = Dataset.from_pandas(augmented_train_df).map(prep_fn)
    dev_ds = Dataset.from_pandas(dev_df).map(prep_fn)
    
    args = TrainingArguments(
        output_dir=f"moe_{name}_results",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        num_train_epochs=3,
        evaluation_strategy="epoch",
        report_to="none"
    )
    
    trainer = Trainer(
        model=model, 
        args=args, 
        train_dataset=train_ds, 
        eval_dataset=dev_ds,
        compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}
    )
    
    trainer.train()
    
    # Get dev set predictions
    preds = trainer.predict(dev_ds)
    all_logits[name] = preds.predictions

# Ensemble Logic: Average Logits
final_logits = (all_logits["deberta"] + all_logits["modernbert"]) / 2
final_preds = np.argmax(final_logits, axis=1)

print("\n--- Final Ensemble Metrics ---")
print(f"Accuracy: {accuracy_score(dev_df['label'], final_preds):.4f}")
print(f"Macro F1: {f1_score(dev_df['label'], final_preds, average='macro'):.4f}")


--- Training MoE with deberta backbone ---


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/25432 [00:00<?, ? examples/s]

KeyboardInterrupt: 